# V2 Ensemble — PLUSVein RL850 Cross-Dataset EER
Loads all 11 trained models, strips their classification heads, embeds every
PLUSVein RL850 image, averages the 11 feature vectors per image (score-level
fusion in embedding space), then computes the ensemble EER, AUC, and operating-
point TAR values. Run cells top-to-bottom. **Edit Cell 1 paths only.**


## Cell 1 · Paths — edit if needed

In [1]:

import os

# ── EDIT THESE ────────────────────────────────────────────────────────────────
MODELS_DIR      = r"F:\Palm Vein Multimodel V2\Model\Saved Models\PalmVein_V2"
PLUSVEIN_DIR    = r"F:\Palm Vein Multimodel V2\External\HandVein PLUSVein\HandVein_RL850_Preprocessed"
NUM_CLASSES     = 150          # classes PVCNN was trained on
IMG_SIZE_STD    = 224          # standard size for all models
IMG_SIZE_INC    = 299          # InceptionV3
OUT_DIR         = r"outputs_ensemble_eer"
IMG_EXTS        = (".png",".jpg",".jpeg",".bmp")
SKIP_INCEPTION  = True         # set False if you want to include InceptionV3

os.makedirs(OUT_DIR, exist_ok=True)
assert os.path.isdir(MODELS_DIR),   f"Models dir not found: {MODELS_DIR}"
assert os.path.isdir(PLUSVEIN_DIR), f"PLUSVein dir not found: {PLUSVEIN_DIR}"
print("Paths OK")


Paths OK


## Cell 2 · Imports and device

In [2]:

import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as T
try:
    import timm
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable,"-m","pip","install","timm","-q"])
    import timm
import numpy as np, cv2, re
from collections import defaultdict
from sklearn.metrics import roc_curve, auc
import itertools, random, os
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))


c:\Users\SMHC\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
GPU: NVIDIA GeForce RTX 5070


## Cell 3 · PVCNN architecture (must match training exactly)

In [3]:

class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(channels, channels // reduction), nn.GELU(),
            nn.Linear(channels // reduction, channels), nn.Sigmoid())
    def forward(self, x):
        w = self.se(x).unsqueeze(-1).unsqueeze(-1)
        return x * w

class MultiScaleBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        mid = out_ch // 2
        self.branch3 = nn.Sequential(nn.Conv2d(in_ch,mid,3,padding=1,bias=False),nn.BatchNorm2d(mid),nn.GELU())
        self.branch5 = nn.Sequential(nn.Conv2d(in_ch,mid,5,padding=2,bias=False),nn.BatchNorm2d(mid),nn.GELU())
        self.se   = SEBlock(out_ch)
        self.proj = nn.Sequential(nn.Conv2d(out_ch,out_ch,1,bias=False),nn.BatchNorm2d(out_ch),nn.GELU())
    def forward(self, x):
        x = torch.cat([self.branch3(x), self.branch5(x)], dim=1)
        x = self.se(x)
        return self.proj(x)

class PVCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.stem   = nn.Sequential(nn.Conv2d(3,32,3,stride=2,padding=1,bias=False),nn.BatchNorm2d(32),nn.GELU())
        self.stage1 = nn.Sequential(MultiScaleBlock(32,64),   nn.MaxPool2d(2))
        self.stage2 = nn.Sequential(MultiScaleBlock(64,128),  nn.MaxPool2d(2))
        self.stage3 = nn.Sequential(MultiScaleBlock(128,256), nn.MaxPool2d(2))
        self.stage4 = nn.Sequential(MultiScaleBlock(256,512), nn.MaxPool2d(2))
        self.pool   = nn.AdaptiveAvgPool2d(1)
        self.head   = nn.Sequential(nn.Flatten(),nn.Linear(512,512),nn.GELU(),nn.Dropout(0.4),nn.Linear(512,num_classes))
    def forward(self, x):
        x = self.stem(x); x = self.stage1(x); x = self.stage2(x)
        x = self.stage3(x); x = self.stage4(x); x = self.pool(x)
        return self.head(x)
    def embed(self, x):
        x = self.stem(x); x = self.stage1(x); x = self.stage2(x)
        x = self.stage3(x); x = self.stage4(x)
        x = self.pool(x); x = x.flatten(1)
        return F.normalize(x, dim=1)

print("PVCNN architecture defined")


PVCNN architecture defined


## Cell 4 · Build and load all 11 models

In [5]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

tf_std = T.Compose([T.ToPILImage(), T.Resize((IMG_SIZE_STD,IMG_SIZE_STD)),
                    T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
tf_inc = T.Compose([T.ToPILImage(), T.Resize((IMG_SIZE_INC,IMG_SIZE_INC)),
                    T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

def build_model(name):
    """Build with NUM_CLASSES so checkpoint shape matches exactly."""
    if name == "ResNet50":
        m = models.resnet50(weights=None); m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    elif name == "ResNet101":
        m = models.resnet101(weights=None); m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    elif name == "VGG16":
        m = models.vgg16(weights=None); m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, NUM_CLASSES)
    elif name == "VGG19":
        m = models.vgg19(weights=None); m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, NUM_CLASSES)
    elif name == "DenseNet121":
        m = models.densenet121(weights=None); m.classifier = nn.Linear(m.classifier.in_features, NUM_CLASSES)
    elif name == "MobileNetV3_Large":
        m = models.mobilenet_v3_large(weights=None); m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, NUM_CLASSES)
    elif name == "ConvNeXt_Tiny":
        m = models.convnext_tiny(weights=None); m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, NUM_CLASSES)
    elif name == "RegNetY_8GF":
        m = models.regnet_y_8gf(weights=None)
        # Adjust classifier layer for different torchvision versions
        if hasattr(m, "head") and hasattr(m.head, "fc"):
            m.head.fc = nn.Linear(m.head.fc.in_features, NUM_CLASSES)
        elif hasattr(m, "fc"):
            m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
        elif hasattr(m, "classifier"):
            cls = m.classifier
            replaced = False
            # try to replace last Linear in classifier
            try:
                for n, module in reversed(list(cls.named_children())):
                    if isinstance(module, nn.Linear):
                        setattr(cls, n, nn.Linear(module.in_features, NUM_CLASSES))
                        replaced = True
                        break
            except Exception:
                pass
            if not replaced:
                # fallback: search whole model for final Linear and replace
                for n, module in reversed(list(m.named_modules())):
                    if isinstance(module, nn.Linear):
                        parts = n.split('.')
                        parent = m
                        for p in parts[:-1]:
                            parent = getattr(parent, p)
                        last = getattr(parent, parts[-1])
                        setattr(parent, parts[-1], nn.Linear(last.in_features, NUM_CLASSES))
                        replaced = True
                        break
            if not replaced:
                raise RuntimeError("Cannot locate RegNet classifier layer to replace")
    elif name == "EfficientNetV2_S":
        m = timm.create_model("tf_efficientnetv2_s", pretrained=False, num_classes=NUM_CLASSES)
    elif name == "InceptionV3":
        m = models.inception_v3(weights=None, aux_logits=False); m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    elif name == "PVCNN":
        m = PVCNN(NUM_CLASSES)
    else:
        raise ValueError(f"Unknown: {name}")
    return m

def load_ckpt(model, name):
    path = os.path.join(MODELS_DIR, f"{name}_best.pth")
    assert os.path.exists(path), f"Checkpoint missing: {path}"
    state = torch.load(path, map_location=DEVICE)
    if isinstance(state, dict) and "model_state_dict" in state: state = state["model_state_dict"]
    elif isinstance(state, dict) and "state_dict" in state: state = state["state_dict"]
    model.load_state_dict(state); model.eval(); model.to(DEVICE)
    return model

def strip_head(model, name):
    if "ResNet" in name:
        dim = model.fc.in_features; model.fc = nn.Identity()
    elif "VGG" in name:
        dim = model.classifier[-1].in_features; model.classifier[-1] = nn.Identity()
    elif "DenseNet" in name:
        dim = model.classifier.in_features; model.classifier = nn.Identity()
    elif "MobileNetV3" in name:
        dim = model.classifier[-1].in_features; model.classifier[-1] = nn.Identity()
    elif "ConvNeXt" in name:
        dim = model.classifier[-1].in_features; model.classifier[-1] = nn.Identity()
    elif "RegNet" in name:
        if hasattr(model, "head") and hasattr(model.head, "fc"):
            dim = model.head.fc.in_features; model.head.fc = nn.Identity()
        elif hasattr(model, "fc"):
            dim = model.fc.in_features; model.fc = nn.Identity()
        elif hasattr(model, "classifier"):
            cls = model.classifier
            found = False
            for n, module in reversed(list(cls.named_children())):
                if isinstance(module, nn.Linear):
                    dim = module.in_features
                    setattr(cls, n, nn.Identity())
                    found = True
                    break
            if not found:
                # search whole model
                for n, module in reversed(list(model.named_modules())):
                    if isinstance(module, nn.Linear):
                        dim = module.in_features
                        parent = model
                        parts = n.split('.')
                        for p in parts[:-1]:
                            parent = getattr(parent, p)
                        setattr(parent, parts[-1], nn.Identity())
                        found = True
                        break
                if not found:
                    raise RuntimeError("Cannot locate RegNet classifier layer to strip")
    elif "EfficientNetV2" in name:
        dim = model.get_classifier().in_features; model.reset_classifier(0)
    elif "InceptionV3" in name:
        dim = model.fc.in_features; model.fc = nn.Identity()
    elif "PVCNN" in name:
        dim = 512
        def embed_forward(x):
            x = model.stem(x); x = model.stage1(x); x = model.stage2(x)
            x = model.stage3(x); x = model.stage4(x)
            return model.pool(x).flatten(1)
        model.forward = embed_forward
    return model, dim

model_names = ["ResNet50","ResNet101","VGG16","VGG19","DenseNet121",
               "MobileNetV3_Large","ConvNeXt_Tiny","RegNetY_8GF",
               "EfficientNetV2_S","PVCNN"]
if not SKIP_INCEPTION:
    model_names.append("InceptionV3")

loaded_models = []
for name in model_names:
    m = build_model(name)
    m = load_ckpt(m, name)
    m, dim = strip_head(m, name)
    tf = tf_inc if name == "InceptionV3" else tf_std
    loaded_models.append((name, m, tf, dim))
    print(f"  {name:25} loaded  embed_dim={dim}")

print(f"\nTotal models loaded: {len(loaded_models)}")


  ResNet50                  loaded  embed_dim=2048
  ResNet101                 loaded  embed_dim=2048
  VGG16                     loaded  embed_dim=4096
  VGG19                     loaded  embed_dim=4096
  DenseNet121               loaded  embed_dim=1024
  MobileNetV3_Large         loaded  embed_dim=1280
  ConvNeXt_Tiny             loaded  embed_dim=768
  RegNetY_8GF               loaded  embed_dim=2016
  EfficientNetV2_S          loaded  embed_dim=1280
  PVCNN                     loaded  embed_dim=512

Total models loaded: 10


## Cell 5 · Scan PLUSVein, parse hand identities

In [6]:

def parse_hand(filename):
    name = os.path.splitext(filename)[0]
    m = re.search(r"(\d{2,4})[_-]?([LR])\b", name) or \
        re.search(r"(\d{2,4}).*?([LR])[-_]?PALMAR", name, re.IGNORECASE)
    if m: return f"{m.group(1)}_{m.group(2).upper()}"
    return name  # fallback

by_hand = defaultdict(list)
for fn in sorted(os.listdir(PLUSVEIN_DIR)):
    if fn.lower().endswith(IMG_EXTS):
        hand = parse_hand(fn)
        by_hand[hand].append(os.path.join(PLUSVEIN_DIR, fn))

hands = [h for h in by_hand if len(by_hand[h]) >= 2]
paths, labels = [], []
for ci, h in enumerate(hands):
    for p in by_hand[h]:
        paths.append(p); labels.append(ci)
labels = np.array(labels)
print(f"Hands: {len(hands)}  |  Images: {len(paths)}")


Hands: 84  |  Images: 420


## Cell 6 · Extract embeddings from all models

In [7]:

all_embs = {}  # model_name -> (N, D) array

for name, model, tf, dim in loaded_models:
    embs = []
    with torch.no_grad():
        for p in tqdm(paths, desc=name, leave=False):
            bgr = cv2.imread(p)
            if bgr is None: raise RuntimeError(f"Cannot read {p}")
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            x = tf(rgb).unsqueeze(0).to(DEVICE)
            e = model(x).squeeze(0).cpu().numpy()
            embs.append(e)
    embs = np.stack(embs)
    # L2-normalise so cosine similarity = dot product
    norms = np.linalg.norm(embs, axis=1, keepdims=True)
    embs = embs / np.clip(norms, 1e-8, None)
    all_embs[name] = embs
    print(f"  {name:25} embeddings: {embs.shape}")

print("All embeddings extracted.")


  ResNet50                  embeddings: (420, 2048)


  ResNet101                 embeddings: (420, 2048)


  VGG16                     embeddings: (420, 4096)


  VGG19                     embeddings: (420, 4096)


  DenseNet121               embeddings: (420, 1024)


  MobileNetV3_Large         embeddings: (420, 1280)


  ConvNeXt_Tiny             embeddings: (420, 768)


  RegNetY_8GF               embeddings: (420, 2016)


  EfficientNetV2_S          embeddings: (420, 1280)


  PVCNN                     embeddings: (420, 512)
All embeddings extracted.


## Cell 7 · Ensemble embedding (average, re-normalise)

In [8]:

# Stack and mean — shape (N, D_max) but each model has different D,
# so we normalise each model's embedding independently and then average.
# The fused similarity is: sim(a,b) = mean_m [ f_m(a)·f_m(b) ]
# This is equivalent to averaging cosine similarities across models.

M = len(all_embs)
print(f"Fusing {M} models by averaging L2-normalised embeddings...")

# Build a fused similarity matrix approach: pair-wise mean cosine similarity
# Memory-efficient: compute per-pair rather than all at once
idx_by = defaultdict(list)
for i, l in enumerate(labels): idx_by[l].append(i)

genuine_pairs = [pair for idxs in idx_by.values() for pair in itertools.combinations(idxs, 2)]
all_pairs = list(itertools.combinations(range(len(paths)), 2))
gset = set(genuine_pairs)
impostor_pairs = [p for p in all_pairs if p not in gset]

MAX_IMP = 200_000
random.seed(42)
if len(impostor_pairs) > MAX_IMP:
    impostor_pairs = random.sample(impostor_pairs, MAX_IMP)

print(f"Genuine pairs: {len(genuine_pairs)}  |  Impostor pairs: {len(impostor_pairs)}")

def ensemble_sim(i, j):
    """Average cosine similarity across all models."""
    sims = []
    for name in all_embs:
        e = all_embs[name]
        sims.append(float(np.dot(e[i], e[j])))
    return np.mean(sims)

print("Computing genuine pair similarities...")
gen_sim = np.array([ensemble_sim(i,j) for i,j in tqdm(genuine_pairs)])
print("Computing impostor pair similarities...")
imp_sim = np.array([ensemble_sim(i,j) for i,j in tqdm(impostor_pairs)])
print("Done.")


Fusing 10 models by averaging L2-normalised embeddings...
Genuine pairs: 840  |  Impostor pairs: 87150
Computing genuine pair similarities...


100%|██████████| 840/840 [00:00<00:00, 46649.04it/s]


Computing impostor pair similarities...


100%|██████████| 87150/87150 [00:01<00:00, 52595.29it/s]

Done.


## Cell 8 · Compute EER, AUC, operating points + save

In [9]:

def compute_eer(gen_s, imp_s):
    y = np.concatenate([np.ones(len(gen_s)), np.zeros(len(imp_s))])
    s = np.concatenate([gen_s, imp_s])
    fpr, tpr, thr = roc_curve(y, s)
    fnr = 1 - tpr
    idx = np.nanargmin(np.abs(fnr - fpr))
    eer = (fpr[idx] + fnr[idx]) / 2 * 100
    return eer, auc(fpr, tpr), fpr, tpr, thr

eer, roc_auc, fpr, tpr, _ = compute_eer(gen_sim, imp_sim)

# TAR at fixed FAR operating points
def tar_at_far(fpr, tpr, target_far):
    idx = np.searchsorted(fpr, target_far)
    if idx >= len(tpr): return tpr[-1]
    return tpr[idx]

tar_001 = tar_at_far(fpr, tpr, 0.001) * 100  # FAR=0.1%
tar_01  = tar_at_far(fpr, tpr, 0.01)  * 100  # FAR=1%
tar_10  = tar_at_far(fpr, tpr, 0.10)  * 100  # FAR=10%

print("\n" + "="*55)
print("V2 ENSEMBLE — PLUSVein RL850 Cross-Dataset Verification")
print("="*55)
print(f"  Models fused       : {M}")
print(f"  EER                : {eer:.2f}%")
print(f"  ROC-AUC            : {roc_auc:.4f}")
print(f"  TAR @ FAR=0.1%     : {tar_001:.2f}%")
print(f"  TAR @ FAR=1.0%     : {tar_01:.2f}%")
print(f"  TAR @ FAR=10.0%    : {tar_10:.2f}%")
print(f"  Genuine mean sim   : {gen_sim.mean():.4f}")
print(f"  Impostor mean sim  : {imp_sim.mean():.4f}")
print("="*55)

# Save summary
with open(os.path.join(OUT_DIR, "ensemble_eer_summary.txt"), "w") as f:
    f.write(f"V2 Ensemble ({M} models) — PLUSVein RL850\n")
    f.write(f"EER: {eer:.3f}%\nAUC: {roc_auc:.4f}\n")
    f.write(f"TAR@FAR=0.1%: {tar_001:.2f}%\n")
    f.write(f"TAR@FAR=1.0%: {tar_01:.2f}%\n")
    f.write(f"TAR@FAR=10.0%: {tar_10:.2f}%\n")
    f.write(f"Genuine mean sim: {gen_sim.mean():.4f}\n")
    f.write(f"Impostor mean sim: {imp_sim.mean():.4f}\n")

# ROC plot
plt.figure(figsize=(7,7))
plt.plot(fpr, tpr, lw=2, color="#1F4E79", label=f"Ensemble {M} models (EER {eer:.1f}%, AUC {roc_auc:.3f})")
plt.plot([0,1],[0,1],"k--",lw=1,label="Random")
plt.xlabel("False Acceptance Rate"); plt.ylabel("True Acceptance Rate")
plt.title(f"V2 Ensemble ROC — PLUSVein RL850")
plt.legend(loc="lower right"); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "ensemble_roc.png"), dpi=150); plt.show()

# Distance distribution
plt.figure(figsize=(7,4))
plt.hist(imp_sim, bins=60, alpha=0.6, label="Impostor", color="#c0392b", density=True)
plt.hist(gen_sim, bins=60, alpha=0.6, label="Genuine",  color="#27ae60", density=True)
plt.title(f"Ensemble Genuine vs Impostor (EER {eer:.1f}%)")
plt.xlabel("Mean cosine similarity"); plt.ylabel("Density"); plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "ensemble_dist.png"), dpi=150); plt.show()

print(f"\nResults saved to: {OUT_DIR}")



V2 ENSEMBLE — PLUSVein RL850 Cross-Dataset Verification
  Models fused       : 10
  EER                : 3.22%
  ROC-AUC            : 0.9939
  TAR @ FAR=0.1%     : 85.60%
  TAR @ FAR=1.0%     : 94.52%
  TAR @ FAR=10.0%    : 98.57%
  Genuine mean sim   : 0.9133
  Impostor mean sim  : 0.7694


C:\Users\SMHC\AppData\Local\Temp\ipykernel_33392\2614696515.py:52: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(os.path.join(OUT_DIR, "ensemble_roc.png"), dpi=150); plt.show()



Results saved to: outputs_ensemble_eer


C:\Users\SMHC\AppData\Local\Temp\ipykernel_33392\2614696515.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(os.path.join(OUT_DIR, "ensemble_dist.png"), dpi=150); plt.show()
